**German Credit Dataset**

# 03.1 - Classic Models Pipeline

**Objectives**
- Build and run machine learning pipelines with different configurations
- Train and evaluate several classical models automatically
- Export performance and fairness results for all model configurations

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
import sys
import warnings
from itertools import product
import time
from aif360.datasets import BinaryLabelDataset
import importlib

In [ ]:
sys.path.append('utils')

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
import preprocessing
import bias_preprocessing
import models
import tuning
import bias_postprocessing
import evaluation

importlib.reload(preprocessing)
importlib.reload(bias_preprocessing)
importlib.reload(models)
importlib.reload(tuning)
importlib.reload(bias_postprocessing)
importlib.reload(evaluation)

## 1. Load Data

In [ ]:
file_path = '../data/processed/german_df_processed_2.csv'
df = pd.read_csv(file_path, sep=r',', header=0)

In [ ]:
df

## 2. Setup: Key Variables, Helper Functions, and Pipeline Construction

### 2.1. Key Variables

In [ ]:
NUMERIC_COLS = [
    'duration_months', 'credit_amount', 'installment_rate', 
    'residence_duration', 'existing_credits_count', 'dependents'
]

CATEGORICAL_COLS = [
    'sex','age_cat','checking_account_status', 'credit_history', 
    'purpose', 'savings_account_status', 'employment_status', 
    'guarantors', 'property', 'other_debts', 'housing', 'job', 
    'own_telephone?', 'foreign_worker?'
]

In [ ]:
TARGET_COLUMN = 'good_client?'
FAVORABLE_LABEL = 1   # 'Good client'
UNFAVORABLE_LABEL = 0 # 'No good client'

SENSITIVE_ATTR = 'sex'
PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Male
UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Female

# SENSITIVE_ATTR = 'foreign_worker?'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Foreign
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 0}] # Local

# SENSITIVE_ATTR = 'age_cat'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Between 24.0 and 49.1 years old
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Other ages

In [ ]:
# 1) Pré-processamento sklearn 
SCALERS = ["none", "standardization"]
ENCODERS = ["none", "one-hot", "label"]  

# 2) Bias preprocessing AIF360
BIAS_PRE = ["none", "reweighing", "disparate-impact-remover"]

# 3) Modelos 
MODELS = ["logistic_regression", "random_forest", "gradient_boosting"]

# 4) Tuning (sem considerar o "grid")
TUNING = ["none", "random"]  

# 5) Bias postprocessing
BIAS_POST = ["none", "calibrate_equalized_odds", "reject_option_classification"]

### 2.2. Helper Functions

In [ ]:
def df_to_aif360(df):
    return BinaryLabelDataset(
        df=df,
        label_names=[TARGET_COLUMN],
        protected_attribute_names=[SENSITIVE_ATTR],
        favorable_label=FAVORABLE_LABEL,
        unfavorable_label=UNFAVORABLE_LABEL
    )

### 2.3. Pipeline Construction

In [ ]:
def run_pipeline(config, df_train, df_test):
    """
    Executes a single pipeline configuration on a fixed train/test split.

      1) Converts df_train/df_test to BinaryLabelDataset (AIF360)
      2) Applies pre-processing bias mitigation (e.g., reweighing)
      3) Separates X/y, removes the sensitive attribute from X (to avoid training directly with it)
      4) Applies standard pre-processing (scaler/encoder) using sklearn
      5) Trains the model (with or without tuning), using sample_weight if available
      6) Generates scores (probabilities) on the test set
      7) Applies post-processing bias mitigation (e.g., reject option / calibrate equalized odds)
      8) Evaluates performance and fairness, and returns the metrics

    Args:
        config (dict): Dictionary with the pipeline choices. Expected keys:
            - 'id'       : textual identifier of the configuration
            - 'scaler'   : type of scaler
            - 'encoder'  : type of encoder 
            - 'bias_pre' : AIF360 pre-processing technique
            - 'model'    : model 
            - 'tuning'   : tuning strategy
            - 'bias_post': AIF360 post-processing technique
        df_train (pd.DataFrame): Training data (including TARGET_COLUMN and the sensitive attribute SENSITIVE_ATTR).
        df_test (pd.DataFrame): Test/validation data (including TARGET_COLUMN and SENSITIVE_ATTR).

    Returns:
        pd.DataFrame: A DataFrame with one row containing the calculated metrics (performance + fairness)
        and the pipeline name/ID.
    """
    print(f"Executando: {config['id']}")

    # ------------------------------------------------------------------
    #  1) Converts df_train/df_test to BinaryLabelDataset
    # ------------------------------------------------------------------
    ds_train = df_to_aif360(df_train)
    ds_test = df_to_aif360(df_test)

    numeric_cols = [c for c in NUMERIC_COLS if c != SENSITIVE_ATTR]
    categorical_cols = [c for c in CATEGORICAL_COLS if c != SENSITIVE_ATTR]

    
    # ------------------------------------------------------------------
    #  2) Applies pre-processing bias mitigation 
    # ------------------------------------------------------------------
    ds_train_bias_proc = bias_preprocessing.apply_bias_preprocessing(
        config['bias_pre'], ds_train, SENSITIVE_ATTR, UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS
    )

    # ------------------------------------------------------------------
    #  3) Separates X/y, removes the sensitive attribute from X 
    # ------------------------------------------------------------------
    X_train_df = pd.DataFrame(ds_train_bias_proc.features, columns=ds_train_bias_proc.feature_names)
    X_test_df  = pd.DataFrame(ds_test.features, columns=ds_test.feature_names)

    X_train_df = X_train_df.drop(columns=[SENSITIVE_ATTR])
    X_test_df  = X_test_df.drop(columns=[SENSITIVE_ATTR])

    y_train = ds_train_bias_proc.labels.ravel()
    sample_weight = ds_train_bias_proc.instance_weights.ravel()

    
    # ------------------------------------------------------------------
    #  4) Applies standard pre-processing
    # ------------------------------------------------------------------
    preprocessor = preprocessing.build_preprocessor(
        numeric_cols, categorical_cols, config['scaler'], config['encoder']
    )
    X_train_proc = preprocessor.fit_transform(
        X_train_df
    )
    X_test_proc = preprocessor.transform(
        X_test_df
    )
    
    # ------------------------------------------------------------------
    #  5) Trains the model using sample_weight if available
    # ------------------------------------------------------------------
    base_model = models.get_model(
        config['model']
    )
    grid_p, random_p = models.get_hyperparameters(config['model'])
    model = tuning.apply_tuning(
        config['tuning'], base_model, grid_p, random_p
    )

    model.fit(
        X_train_proc, 
        y_train, 
        sample_weight=sample_weight
    )
 
    # ------------------------------------------------------------------
    #  6) Generates scores (probabilities) on the test set
    # ------------------------------------------------------------------
    ds_pred = ds_test.copy()
    ds_pred.scores = model.predict_proba(X_test_proc)[:, 1].reshape(-1, 1)
    
    # ------------------------------------------------------------------
    #  7) Applies post-processing bias mitigation
    # ------------------------------------------------------------------
    ds_pred_mitigated = bias_postprocessing.apply_bias_postprocessing(
        config['bias_post'], ds_test, ds_pred, UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS
    )

    # ------------------------------------------------------------------
    #  8) Evaluates performance and fairness, and returns the metrics
    # ------------------------------------------------------------------
    return evaluation.evaluate_pipeline(ds_test, ds_pred_mitigated, UNPRIVILEGED_GROUPS, PRIVILEGED_GROUPS, config['id'])

In [ ]:
def run_cv_pipeline(config, df, n_splits=5, random_state=42, shuffle=True):
    """
    Executes Cross-Validation (StratifiedKFold) for a single pipeline configuration and aggregates the metrics.

      1) Splits the complete DataFrame 'df' into K stratified folds by TARGET_COLUMN
      2) For each fold:
           - Separates df_train_fold and df_test_fold
           - Executes run_pipeline(config, df_train_fold, df_test_fold)
           - Stores the single row of metrics for that fold
      3) Aggregates the metrics across all folds:
           - mean_* : mean across folds
           - std_*  : standard deviation across folds

    Args:
        config (dict): Pipeline configuration (id, scaler, encoder, bias_pre, model, tuning, bias_post).
        df (pd.DataFrame): Complete dataset, containing TARGET_COLUMN and SENSITIVE_ATTR.
        n_splits (int): Number of folds (K).
        random_state (int): Seed (only has effect if shuffle=True).
        shuffle (bool): If True, shuffles the data before creating the folds.

    Returns:
        pd.DataFrame: A DataFrame with one row containing the aggregated metrics (mean_*, std_*) and config metadata.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    y = df[TARGET_COLUMN].values

    fold_rows = []  # Dictionary to store results from each fold

    # ------------------------------------------------------------------
    # 1) Executes each fold
    # ------------------------------------------------------------------
    for fold, (train_idx, test_idx) in enumerate(skf.split(df, y), start=1):
        df_train_fold = df.iloc[train_idx].copy()
        df_test_fold  = df.iloc[test_idx].copy()

        df_fold = run_pipeline(config, df_train=df_train_fold, df_test=df_test_fold)

        row = df_fold.iloc[0].to_dict()
        row.pop("Pipeline", None)  # Remove text field (not for aggregation)
        fold_rows.append(row)

    # ------------------------------------------------------------------
    # 2) Aggregates (mean/std) across folds
    # ------------------------------------------------------------------
    df_tmp = pd.DataFrame(fold_rows)

    means = df_tmp.mean(numeric_only=True)
    stds  = df_tmp.std(numeric_only=True)

    summary_row = {"config_id": config["id"], "n_splits": n_splits}
    for col, val in means.items():
        summary_row[f"mean_{col}"] = float(val)
    for col, val in stds.items():
        summary_row[f"std_{col}"] = float(val)

    df_summary = pd.DataFrame([summary_row])
    return df_summary

## 3. Pipeline Execution

### 3.1. Single Model Test

In [ ]:
df_summary  = run_cv_pipeline(
    config={
        "id": "German Credit Single Model Test",
        "scaler": "standardization",
        "encoder": "label",
        "bias_pre": "none",
        "model": "gradient_boosting",
        "tuning": "none",
        "bias_post": "none",
    },
    df=df,
    n_splits=3
)


In [ ]:
df_summary

### 3.2. All Models

In [ ]:
def run_all_combinations(
    df,
    n_splits=5,
    random_state=42,
    shuffle=True,
    fail_fast=False
):
    """
    Runs ALL possible pipeline combinations with cross-validation and returns one summary row (mean/std) per configuration.

      1) Generates all possible combinations of pipeline elements (scaler, encoder, bias mitigation, model, tuning, etc.)
      2) For each configuration:
           - Executes cross-validation with run_cv_pipeline
           - Stores the summary metrics
           - Handles and logs any errors 
      3) Aggregates the results:
           - Returns a DataFrame with summarized results from all successful configurations
           - Returns a DataFrame with information about any errors encountered

    Args:
        df (pd.DataFrame): Complete dataset including TARGET_COLUMN and SENSITIVE_ATTR.
        n_splits (int): Number of CV folds.
        random_state (int): Seed used when shuffling the data for reproducibility.
        shuffle (bool): If True, shuffles the data before fold split.
        fail_fast (bool): If True, stops at the first error encountered during execution.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]:
            - results_cv_df: DataFrame with one summary row (mean/std) per pipeline configuration.
            - errors_df: DataFrame listing configurations and errors encountered during execution.
    """
    # ------------------------------------------------------------------
    # 1) Generate all pipeline configuration combinations
    # ------------------------------------------------------------------
    configs = []
    for scaler, encoder, bias_pre, model, tuning_method, bias_post in product(
        SCALERS, ENCODERS, BIAS_PRE, MODELS, TUNING, BIAS_POST
    ):
        configs.append({
            "id": f"{model}|{scaler}|{encoder}|pre={bias_pre}|tune={tuning_method}|post={bias_post}",
            "scaler": scaler,
            "encoder": encoder,
            "bias_pre": bias_pre,
            "model": model,
            "tuning": tuning_method,
            "bias_post": bias_post,
        })

    total = len(configs)
    print(f"Total combinations: {total} | n_splits={n_splits}")

    summaries = []
    errors = []
    start_all = time.time()

    # ------------------------------------------------------------------
    # 2) Run CV for each configuration and collect either results or errors
    # ------------------------------------------------------------------
    for i, cfg in enumerate(configs, start=1):

        print(f"\n[{i}/{total}] Running: {cfg['id']}")

        try:
            df_summary = run_cv_pipeline(
                config=cfg,
                df=df,
                n_splits=n_splits,
                random_state=random_state,
                shuffle=shuffle
            )
            summaries.append(df_summary)

        except Exception as e:
            print(f"[ERROR] {cfg['id']}")
            print(f"       {type(e).__name__}: {e}")

            errors.append({
                "config_id": cfg["id"],
                "error_type": type(e).__name__,
                "error_message": str(e),
            })

            if fail_fast:
                print("[FAIL_FAST] Stopping after the first failure.")
                break

    total_time = time.time() - start_all
    print("\n" + "#" * 90)
    print(f"Cross-validation completed in {total_time/60:.2f} min")
    print(f"Successes: {len(summaries)} | Failures: {len(errors)}")
    print("#" * 90)

    # ------------------------------------------------------------------
    # 3) Aggregate: concatenate summaries, collect errors, and return
    # ------------------------------------------------------------------
    results_cv_df = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    errors_df = pd.DataFrame(errors)
    return results_cv_df, errors_df

In [ ]:
results, errors = run_all_combinations(df=df, n_splits=3)

In [ ]:
results.sort_values("mean_F1-Score", ascending=False).head(20)

In [ ]:
errors

## 3. Exporting Results

In [ ]:
file_out_path = '../data/results'

results.to_csv(file_out_path + '/classic_models_results.csv', index=False)